<table>
  <tr>
    <td><div align="left"><font size="30">Furuta pendulum</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the rotary inverted (Furuta) pendulum using *bdsim***

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

A rotary inverted pendulum, or Furuta pendulum, has a horizontal arm that is driven by a motor to rotate about a vertical axis, and a pendulum attached to the end of the arm that swings freely in the vertical plane perpendicular to the arm.  The motor torque $\tau$ is the only input and the aim is to balance the pendulum upright by moving the arm.  Like the cart-pole it is underactuated and the upright position is an unstable equilibrium, but it has no end stops: the arm can turn indefinitely, so the pendulum can be swung up from hanging and then caught.  The arm moves on a circle, not a straight line, which makes the problem three-dimensional and the equations of motion more coupled than those of the cart-pole.

The arm angle $\theta$ is measured about the vertical axis, counter-clockwise seen from above, and $\tau$ is the torque on the arm about that axis.  The pendulum angle $\alpha$ is measured from the **downward** vertical, and is positive when the pendulum swings in the direction of increasing $\theta$.  So $\alpha = 0$ is the stable hanging equilibrium and $\alpha = \pi$ is the unstable upright one.  This is the convention of the swing-up literature, and it is the opposite of the cart-pole notebook, where $\theta = 0$ was upright.  It costs little in the control design: we linearize about $\alpha = \pi$ and use $\beta = \alpha - \pi$ for the angle from upright.

We assume that:

* the arm is a uniform rod of mass $m_r$ and length $L_r$, pivoted at one end on the motor shaft, so its moment of inertia about the shaft is $J_r = \frac{1}{3} m_r L_r^2$,
* the pendulum is a uniform rod of mass $m_p$ and length $L_p$, pivoted at one end on the tip of the arm, so its centre of mass is $l_p = L_p/2$ from the pivot and its moment of inertia about its centre of mass is $J_p = \frac{1}{12} m_p L_p^2$,
* there is viscous friction on the arm shaft, with coefficient $D_r$, and at the pendulum pivot, with coefficient $D_p$, and
* the actuator is perfect: the torque $\tau$ is whatever we command, applied directly to the arm, with no limit and no dynamics.  The motor and its electronics are left as an exercise.

The equations of motion follow from the Euler-Lagrange method.  Writing $I_p = J_p + m_p l_p^2 = \frac{1}{3} m_p L_p^2$ for the inertia of the pendulum about its pivot, the kinetic energy is
$$
T = \frac{1}{2} \left( J_r + m_p L_r^2 + I_p \sin^2 \alpha \right) \dot{\theta}^2 + m_p L_r l_p \cos \alpha \, \dot{\theta} \dot{\alpha} + \frac{1}{2} I_p \dot{\alpha}^2 \qquad (1)
$$
The first term is the arm, together with the pendulum carried round by it: the further the pendulum swings out from the vertical axis, the more it adds to the inertia the motor sees.  The second term couples the arm to the pendulum, and the third is the pendulum swinging about its pivot.  The potential energy, taking it to be zero with the pendulum hanging, is
$$
V = m_p g l_p \left( 1 - \cos \alpha \right) \qquad (2)
$$
The generalized forces are $\tau - D_r \dot{\theta}$ on the arm and $-D_p \dot{\alpha}$ on the pendulum, and the Euler-Lagrange equations for $\theta$ and $\alpha$ are
$$
\left( J_r + m_p L_r^2 + I_p \sin^2 \alpha \right) \ddot{\theta} + m_p L_r l_p \cos \alpha \, \ddot{\alpha} + 2 I_p \sin \alpha \cos \alpha \, \dot{\theta} \dot{\alpha} - m_p L_r l_p \sin \alpha \, \dot{\alpha}^2 = \tau - D_r \dot{\theta} \qquad (3)
$$
$$
m_p L_r l_p \cos \alpha \, \ddot{\theta} + I_p \ddot{\alpha} - I_p \sin \alpha \cos \alpha \, \dot{\theta}^2 + m_p g l_p \sin \alpha = - D_p \dot{\alpha} \qquad (4)
$$
These are two linear equations in $\ddot{\theta}$ and $\ddot{\alpha}$, which we solve for the accelerations to feed a pair of integrator chains.  As a sanity check we will compute the total energy $E = T + V$, which can change only through the torque and the friction
$$
\dot{E} = \tau \dot{\theta} - D_r \dot{\theta}^2 - D_p \dot{\alpha}^2 \qquad (5)
$$
so once the torque is removed it can only decrease.

## Parameters

The parameter values used in this notebook, for a small desktop unit, are

| Quantity | Value |
| --- | --- |
| Mass of arm ($m_r$) | 0.095 kg |
| Length of arm, shaft to pendulum pivot ($L_r$) | 0.085 m |
| Viscous friction coefficient of arm ($D_r$) | 0.0015 N m s/rad |
| Mass of pendulum ($m_p$) | 0.024 kg |
| Length of pendulum ($L_p$) | 0.13 m |
| Viscous friction coefficient of pendulum ($D_p$) | 0.0005 N m s/rad |
| Gravitational acceleration ($g$) | 9.81 m/s² |
| Inertia of arm about the shaft ($J_r = \frac{1}{3} m_r L_r^2$) | $2.29 \times 10^{-4}$ kg m² |
| Centre of mass of pendulum ($l_p = L_p / 2$) | 0.065 m |
| Inertia of pendulum about its centre of mass ($J_p = \frac{1}{12} m_p L_p^2$) | $3.38 \times 10^{-5}$ kg m² |
| Inertia of pendulum about its pivot ($I_p = \frac{1}{3} m_p L_p^2$) | $1.35 \times 10^{-4}$ kg m² |

### Sources and assumptions

The masses, lengths and friction coefficients are from Table 1 of [1], for a Quanser QUBE-Servo 2 with its pendulum.  The pendulum mass and length agree with the values identified experimentally in [2], Table 3: 0.024 kg and 0.128 m.  The equations of motion are derived here, not copied.  Dropping the rotation of the pendulum about the vertical axis, which is the part $J_p \sin^2 \alpha$ of the $I_p \sin^2 \alpha$ term in (1), they reduce to the form that Quanser publishes for its rotary pendulum module, as reproduced in the lab handout [3].

The sources are not consistent with each other, or internally, in several ways, and we have made these choices:

* **Arm inertia.** Table 1 of [1] gives $J_r = 5.7 \times 10^{-5}$ kg m² "about the center of mass", which is $m_r L_r^2 / 12$, the value for a rod about its centre.  The equations of motion need the inertia about the shaft.  We take the arm to be a uniform rod pivoted at its end, $J_r = m_r L_r^2 / 3 = 2.29 \times 10^{-4}$ kg m², which is four times larger.  It matters, since the arm and the pendulum together present about $4 \times 10^{-4}$ kg m² to the motor, rather than about $2.3 \times 10^{-4}$.
* **Pendulum inertia.** The $J_p = 3.4 \times 10^{-5}$ kg m² about the centre of mass in [1] is $m_p L_p^2 / 12 = 3.38 \times 10^{-5}$ for a uniform rod, which we compute from the mass and length.
* **The arm differs between the two QUBE sources**: 0.095 kg and 0.085 m in [1], 0.053 kg and 0.086 m in [2].  We use [1].  The pendulums agree.
* **The published equations are not used.** Quanser's form [3] omits the rotation of the pendulum about the vertical axis, and has $m_p l_p^2 \sin^2 \alpha$ where we have $I_p \sin^2 \alpha$.  For a thin rod the part omitted is a third the size of the part kept, which matters when the arm moves fast, as it does in a swing-up.  The equations in [1], its (19) and (20), also have opposite signs on the coupling between $\ddot{\theta}$ and $\ddot{\alpha}$.  A mass matrix must be symmetric, and we could not reproduce the state-space matrix printed in [1] from a consistent model with its own parameters.  We take only its parameters.
* **Friction.** $D_r$ and $D_p$ are "equivalent viscous damping" coefficients as seen at each axis.  Real friction is not purely viscous.
* **The values are not measurements** of a unit of ours.  An exercise suggests how to identify the parameters of your own pendulum.

**References**

[1] T. Räsänen and V.-P. Pyrhönen, "State feedback control of a rotary inverted pendulum", in *Automaatiopäivät23*, Oulu, 15-16 May 2019, Finnish Society of Automation, ISBN 978-952-5183-54-2.  [PDF](https://www.automaatioseura.fi/site/assets/files/2398/procap2352_rasanenpyrhonen.pdf)

[2] H. N. Le, P. V. Dang, A.-D. Pham and N. T. Vo, "System identifications of a 2DOF pendulum controlled by QUBE-servo and its unwanted oscillation factors", *Archive of Mechanical Engineering*, vol. 67, no. 4, 2020, [doi:10.24425/ame.2020.131699](https://doi.org/10.24425/ame.2020.131699).  [PDF](https://journals.pan.pl//Content/115031/PDF/AME_2020_131699.pdf)

[3] Georgia Institute of Technology, AE4610, "Lab 4: Rotary Inverted Pendulum, A & B. Modeling and Balance Control", a lab handout for Quanser's Rotary Pendulum module on the SRV02 servo.  [Web page](https://gtae.gitbook.io/ae4610/lab-4-rotary-inverted-pendulum/a-and-b.-modeling-and-balance-control), accessed 14 November 2026.

# Implementation with *bdsim*

We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.  The arm and pendulum move
quickly, so we set `animation_rate=25`: the scope and animation are refreshed 25 times per
second of simulated time.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=25)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
m_r = 0.095               # mass of arm (kg)
L_r = 0.085               # length of arm, shaft to pendulum pivot (m)
D_r = 0.0015              # viscous friction coefficient of arm (N m s/rad)
m_p = 0.024               # mass of pendulum (kg)
L_p = 0.13                # length of pendulum (m)
D_p = 0.0005              # viscous friction coefficient of pendulum (N m s/rad)
g = 9.81                  # gravity (m/s^2)

# derived parameters, for uniform rods
J_r = m_r * L_r**2 / 3    # arm inertia about the shaft (kg m^2)
l_p = L_p / 2             # pivot to pendulum centre of mass (m)
J_p = m_p * L_p**2 / 12   # pendulum inertia about its centre of mass (kg m^2)
I_p = J_p + m_p * l_p**2  # pendulum inertia about its pivot (kg m^2)

The system can be modeled as two chains of integrators.  The first integrates the arm's angular acceleration $\ddot{\theta}$ to angular velocity $\dot{\theta}$ to angle $\theta$.  The initial angle and velocity are zero.

In [ ]:
theta_dot = bd.INTEGRATOR(name="theta_dot")
theta = theta_dot >> bd.INTEGRATOR(name="theta")

Note that we are using the right-shift operator to implicitly **connect** the output of the `theta_dot` block to the input of the next integrator, and the result of that expression is assigned to `theta`.

The other chain integrates the pendulum's angular acceleration $\ddot{\alpha}$ to angular velocity $\dot{\alpha}$ to angle $\alpha$.  The initial velocity is zero and the pendulum starts upright, at $\alpha = \pi$.

In [ ]:
alpha_dot = bd.INTEGRATOR(name="alpha_dot")
alpha = alpha_dot >> bd.INTEGRATOR(name="alpha", x0=math.pi)  # the pendulum starts upright

We use a pair of function blocks to compute the sine and cosine of the pendulum angle.  A third computes $\beta = \alpha - \pi$, the angle from upright, which we need for control.  The pendulum can swing right round, so $\alpha$ is not confined to $[0, 2\pi)$ and we wrap $\beta$ into $[-\pi, \pi)$.  Without this a pendulum that had swung round once would be seen by the controller as a long way from upright, when it is not.

In [ ]:
sin_alpha = alpha >> bd.FUNCTION(math.sin, name="sin(alpha)")
cos_alpha = alpha >> bd.FUNCTION(math.cos, name="cos(alpha)")
beta = alpha >> bd.FUNCTION(lambda a: a % (2 * math.pi) - math.pi, name="beta")  # angle from upright

The torque is the only input.  The system starts in an unstable equilibrium, with the pendulum upright and nothing moving, so nothing would happen without a disturbance.  We apply a pulse of 0.02 N m for $t \in [1, 1.2]$ s to disturb it.

The applied torque is the pulse minus a state-feedback term $\mathbf{K}x$.  We include the feedback now, but with a gain of zero, so
the system is open loop.  Once we have designed a controller we will set the gain, without changing anything else in the model.  The state vector for the feedback is assembled by a `MUX` block from the arm velocity and angle, the pendulum velocity, and the angle from upright $\beta$.  The gain block is given the name `K` so that we can find it later.

In [ ]:
state = bd.MUX(4, inputs=[theta_dot, theta, alpha_dot, beta], name="state")  # the state vector, in this order
Kgain = bd.GAIN(np.zeros((1, 4)), premul=True, name="K", inputs=[state])  # state feedback gain, zero for now

tau = bd.PIECEWISE((0, 0), (1, 0.02), (1.2, 0), name="torque") - Kgain  # torque on the arm (N m)

Equations (3) and (4) are two linear equations in the accelerations,
$$
\begin{pmatrix} a_{11} & a_{12} \\ a_{12} & I_p \end{pmatrix}
\begin{pmatrix} \ddot{\theta} \\ \ddot{\alpha} \end{pmatrix} =
\begin{pmatrix} b_1 \\ b_2 \end{pmatrix}
$$
with $a_{11} = J_r + m_p L_r^2 + I_p \sin^2 \alpha$ and $a_{12} = m_p L_r l_p \cos \alpha$, which we compute first, along with the determinant of the mass matrix

In [ ]:
a11 = J_r + m_p * L_r**2 + I_p * sin_alpha**2
a12 = m_p * L_r * l_p * cos_alpha
det = a11 * I_p - a12**2  # determinant of the mass matrix

and then the right-hand sides, which hold the torque, the friction, and the centrifugal, Coriolis and gravity terms

In [ ]:
b1 = (
    tau
    - D_r * theta_dot
    - 2 * I_p * sin_alpha * cos_alpha * theta_dot * alpha_dot
    + m_p * L_r * l_p * sin_alpha * alpha_dot**2
)
b2 = -D_p * alpha_dot + I_p * sin_alpha * cos_alpha * theta_dot**2 - m_p * g * l_p * sin_alpha

The solution of the two equations gives the arm acceleration and the pendulum acceleration, which are the inputs to the two chains of integrators

In [ ]:
theta_dot[0] = (I_p * b1 - a12 * b2) / det  # arm acceleration (rad/s^2)
alpha_dot[0] = (a11 * b2 - a12 * b1) / det  # pendulum acceleration (rad/s^2)

As a cross check we compute the total energy of the system $E$ (1)-(2).  We display the two angles on one scope, and the energy and the torque on another.  They are small numbers in SI units, so we scale them to millijoules and millinewton metres, which puts them on a similar scale to each other.
After the torque pulse, which adds energy, we should see the total energy of the system decrease monotonically (5).  The friction on the arm and the pendulum removes it.

In [ ]:
KE = (
    1 / 2 * (J_r + m_p * L_r**2 + I_p * sin_alpha**2) * theta_dot**2
    + m_p * L_r * l_p * cos_alpha * theta_dot * alpha_dot
    + 1 / 2 * I_p * alpha_dot**2
)  # kinetic energy (J)
PE = m_p * g * l_p * (1 - cos_alpha)  # potential energy (J), zero with the pendulum hanging down
E = KE + PE  # total energy (J)

# display the system state evolution on scopes, and an animation
scope = bd.SCOPE(
    inputs=[theta, alpha - math.pi],
    labels=(r"$\theta$ (rad)", r"$\alpha - \pi$ (rad)"),
    loc="lower right",
)
energy_scope = bd.SCOPE(
    inputs=[1000 * tau, 1000 * E],
    labels=(r"$\tau$ (mN m)", "E (mJ)"),
    loc="upper right",
)

To include a graphical animation of our Furuta pendulum we need to create two functions: graphics setup,
and graphics update.  The system is three-dimensional, so we ask the `ANIMATION` block for 3D axes with `projection="3d"` and draw
the parts as solids: the motor hub is a cylinder, and the arm and the pendulum are rectangular prisms, built from Matplotlib `Poly3DCollection` polygons.  Matplotlib is not a 3D renderer, and it sorts the polygons of
a scene by depth as it draws them, which can put the large floor in front of the hub.  We switch that off, `ax.computed_zorder = False`, and order the
parts ourselves.  The camera never moves, so this is easy: the arm sits on top of the hub, so it is always in front of it, and only the
pendulum, which hangs below the arm, can be hidden by the hub, when the arm points away from the camera.  As before, this kind of code is tedious to write, and it can be outsourced to a coding agent.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

AZIM, ELEV = -58, 22  # camera direction (degrees)


def prism(p0, p1, up, w, h):
    # the six faces of a rectangular prism from p0 to p1, with a section w wide and h high
    axis = (p1 - p0) / np.linalg.norm(p1 - p0)
    u = up - axis * (axis @ up)  # the "up" direction of the section, perpendicular to the axis
    u /= np.linalg.norm(u)
    v = np.cross(axis, u)
    corners = [(1, 1), (1, -1), (-1, -1), (-1, 1)]
    a = [p0 + sh * u * h / 2 + sw * v * w / 2 for sh, sw in corners]
    b = [p1 + sh * u * h / 2 + sw * v * w / 2 for sh, sw in corners]
    sides = [[a[i], a[(i + 1) % 4], b[(i + 1) % 4], b[i]] for i in range(4)]
    return [np.array(f) for f in [a, b] + sides]


def cylinder(radius, z0, z1, n=32):
    # the faces of a vertical cylinder: n side quadrilaterals and two end caps
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    lo = np.c_[radius * np.cos(t), radius * np.sin(t), np.full(n, z0)]
    hi = np.c_[radius * np.cos(t), radius * np.sin(t), np.full(n, z1)]
    sides = [np.array([lo[i], lo[(i + 1) % n], hi[(i + 1) % n], hi[i]]) for i in range(n)]
    return sides + [lo, hi]


def furuta_init(self, fig, ax, L_r=L_r, L_p=L_p):
    ax.set_xlim(-0.12, 0.12)
    ax.set_ylim(-0.12, 0.12)
    ax.set_zlim(-0.14, 0.15)
    ax.set_box_aspect((1, 1, 1.2))  # the same scale on all three axes
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_axis_off()
    ax.computed_zorder = False  # we order the artists ourselves, see furuta_update

    def solid(faces, color):
        c = Poly3DCollection(faces, facecolors=color, edgecolors="k", linewidths=0.3, shade=True)
        ax.add_collection3d(c)
        return c

    floor = [[(-0.1, -0.1, -0.14), (0.1, -0.1, -0.14), (0.1, 0.1, -0.14), (-0.1, 0.1, -0.14)]]
    z = np.array([0, 0, 1.0])
    self.floor = solid(floor, "#cccccc")  # a floor, for reference
    self.hub = solid(cylinder(0.02, -0.07, 0.0), "#0072B2")  # the motor hub, which never moves
    self.arm = solid(prism(np.zeros(3), np.array([L_r, 0, 0]), z, 0.015, 0.008), "#E69F00")
    self.pend = solid(prism(np.zeros(3), np.array([0, 0, -L_p]), np.array([1.0, 0, 0]), 0.008, 0.004), "#D55E00")
    self.floor.set_zorder(0)
    self.hub.set_zorder(1)
    self.arm.set_zorder(2)  # the arm is on top of the hub, so it is always in front of it


def furuta_update(self, t, u, L_r=L_r, L_p=L_p):
    theta, alpha = u[0], u[1]
    e_r = np.array([np.cos(theta), np.sin(theta), 0])  # along the arm
    e_t = np.array([-np.sin(theta), np.cos(theta), 0])  # in the direction of increasing theta
    z = np.array([0, 0, 1.0])
    pivot = np.array([0, 0, 0.004])  # the arm lies on top of the hub
    tip = pivot + L_r * e_r
    end = tip + L_p * (np.sin(alpha) * e_t - np.cos(alpha) * z)  # alpha = 0 is hanging down
    self.arm.set_verts(prism(pivot, tip, z, 0.015, 0.008))
    self.pend.set_verts(prism(tip, end, e_r, 0.008, 0.004))
    # the hub can hide the pendulum, but only when the arm points away from the camera
    toward_camera = np.array([np.cos(np.radians(AZIM)), np.sin(np.radians(AZIM)), 0])
    self.pend.set_zorder(3 if tip @ toward_camera > 0 else 0.5)

These two functions are passed to an `ANIMATION` block driven by the outputs of the `theta` and `alpha` blocks.  These inputs
are the elements of `u` passed to `furuta_update`.

In [ ]:
anim = bd.ANIMATION(
    init=furuta_init, update=furuta_update, nin=2, inputs=[theta, alpha], projection="3d"
)

Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

# bd.report_summary()

The model comprises 71 blocks, but the majority were created implicitly (block names are prefixed by `_`) by the arithmetic expressions. A full list of all blocks in the model can be displayed by uncommenting the line above, and rerunning the cell. To create all these blocks and connect them would be both tedious and error prone.  The power of *bdsim* is that it lets
you move effortlessly between expression in terms of blocks or using Python expressions.

We simulate the open-loop system for 5 seconds.  The pendulum starts upright and nothing happens until the torque pulse at $t = 1$ s, which spins the arm up to about 6.7 rad/s and disturbs the pendulum.
It tips over, passing the horizontal about a quarter of a second after the pulse starts, swings through the hanging position and up to about $65°$ beyond it, and then swings back and forth, with rapidly decreasing amplitude, to come to rest hanging.
With nothing driving it, friction brings the arm to rest after it has turned about 2.7 rad (153°).

The energy scope confirms the sanity check.  The system starts with $E = 2 m_p g l_p = 0.0306$ J, the potential energy of the upright pendulum.  That is a small number because the pendulum is light, 24 g raised 0.13 m.  The pulse adds about 0.006 J, and from then on the energy falls monotonically as the friction removes it (5),
to less than 1% of its starting value about a second after the pulse has ended.

In [ ]:
Kgain.K = np.zeros((1, 4))  # no feedback, in case the loop has been closed since
out = sim.run(bd, T=5, watch=[tau, theta, alpha])  # simulate for 5s
print(out)

# Linearization

We can linearize this non-linear system around an equilibrium operating point, here the upright pendulum with the arm at rest.
An operating point is a state together with the input that holds it there, so we give both to `linearize`: the arm angle and both velocities zero,
the pendulum angle $\alpha = \pi$, and no torque.  The input is the output of the `torque` block, and the outputs are the arm angle and the pendulum angle.

In [ ]:
Kgain.K = np.zeros((1, 4))  # the plant alone, with no feedback
lin = bd.linearize(x=[0, 0, 0, math.pi], u=[0], inputs=["torque"], outputs=[theta, alpha], full=True)
print(lin)

The states are in the order $(\dot{\theta}, \theta, \dot{\alpha}, \alpha)$, the order of the blocks as they were added to the diagram.  For small $\beta = \alpha - \pi$
we have $\sin \alpha \approx -\beta$ and $\cos \alpha \approx -1$, and the terms in $\dot{\theta}^2$, $\dot{\alpha}^2$ and $\dot{\theta}\dot{\alpha}$ are second order, so equations (3) and (4) become

$$
\begin{pmatrix} J_r + m_p L_r^2 & -m_p L_r l_p \\ -m_p L_r l_p & I_p \end{pmatrix}
\begin{pmatrix} \ddot{\theta} \\ \ddot{\beta} \end{pmatrix} =
\begin{pmatrix} \tau - D_r \dot{\theta} \\ m_p g l_p \beta - D_p \dot{\beta} \end{pmatrix}
$$

Writing $\mathbf{M}$ for the mass matrix, $\mathbf{A}$ and $\mathbf{B}$ follow by multiplying the right-hand side by $\mathbf{M}^{-1}$.  We can compare them with the numerical result

In [ ]:
M = np.array([[J_r + m_p * L_r**2, -m_p * L_r * l_p], [-m_p * L_r * l_p, I_p]])
Minv = np.linalg.inv(M)
mgl = m_p * g * l_p

A_hand = np.array([
    [-D_r * Minv[0, 0], 0, -D_p * Minv[0, 1], mgl * Minv[0, 1]],
    [1, 0, 0, 0],
    [-D_r * Minv[1, 0], 0, -D_p * Minv[1, 1], mgl * Minv[1, 1]],
    [0, 0, 1, 0],
])
B_hand = np.array([[Minv[0, 0]], [0], [Minv[1, 0]], [0]])
print(np.allclose(lin.A, A_hand, rtol=1e-4), np.allclose(lin.B, B_hand, rtol=1e-4))
print(lin.poles)

There is one pole in the right-half plane, at $s = +9.99$ rad/s.  This is the falling pendulum that we saw tip over in the open-loop run.  It grows by a factor of $e$ every
$1/9.99 = 0.10$ s, so a controller has very little time to react: this is a faster instability than the cart-pole's 4.5 rad/s.  The pole at $s = 0$ is the integrator that turns arm velocity into arm angle: nothing restores the arm to
its angle.  The other two are stable.

The matrix $\mathbf{B}$ shows that a positive torque accelerates the arm in the direction of increasing $\theta$ by about 3700 rad/s² per N m, and also increases $\beta$, at about 3600 rad/s² per N m.  Since $\beta > 0$ means the pendulum leans against the direction of the arm's motion,
this is the pivot being accelerated out from under the pendulum, which is the same effect as in the cart-pole.

# Control

## Control design

We will design a linear controller using the Python Control package, which is not a dependency of `bdsim`

```
$ pip install control
```

and build an LQ state-feedback controller to hold the pendulum upright and the arm at its starting angle, despite a disturbance torque.
We start by exporting the *bdsim* linearization as a Python Control state-space object, with its `ss()` method.  The state names are copied across,
and this is a reminder of the order of the states.

In [ ]:
import control as ct

sys = lin.ss()  # export as a Python Control state-space object, with the state names
print(sys.state_labels)

To design an LQ regulator we need to define the $Q$ (penalizes state errors) and $R$ (penalizes control effort) matrices.
$Q$ must be positive semi-definite and $R$ must be positive definite -- both are typically chosen to be diagonal.
A good starting point is Bryson's rule, which sets each weight to the inverse square of the largest value that we are prepared to tolerate: $Q_{ii} = 1 / x_{i,\max}^2$ and $R = 1 / u_{\max}^2$.
We tolerate the arm being up to 0.5 rad (29°) from its starting angle and the pendulum within 0.2 rad (11°) of upright, and put no weight on the velocities.

The largest torque is limited by the motor.  An estimate of the stall torque is $k_t V_{\max} / R_m$, from the motor's torque constant $k_t = 0.042$ N m/A and resistance $R_m = 8.4$ Ω ([1], Table 1)
and the 18 V supply ([2], Table 1).  That is 0.09 N m, and the real maximum is less, since the back-emf reduces the current when the arm is moving.

In [ ]:
R_m, k_t, V_max = 8.4, 0.042, 18.0  # motor resistance (ohm), torque constant (N m/A), supply voltage (V)
tau_max = k_t * V_max / R_m  # stall torque (N m), an upper bound on the torque the motor can supply
print(tau_max)

Q = np.diag([0, 1 / 0.5**2, 0, 1 / 0.2**2])  # Bryson's rule, weights for [theta_dot, theta, alpha_dot, alpha]
R = np.array([[1 / tau_max**2]])  # weight for control input, the torque

To find the optimal state-feedback $\tau = -\mathbf{K}x$ we need to solve the algebraic Riccati equation

In [ ]:
K, S, poles_cl = ct.lqr(sys, Q, R)
print(K)
print(poles_cl)

where `K` is the optimal state-feedback gain matrix, `S` is the solution to the algebraic Riccati equation, and `poles_cl`
are the eigenvalues (poles) of the closed-loop system ($A - BK$).

To solve the algebraic Riccati equation the package uses an efficient Fortran-based package `slicot` if it is available (via the Python wrapper [slycot](https://github.com/python-control/Slycot)) otherwise it falls back to a pure-Python solution implemented in SciPy.

The gain matrix $K$ has one row per input and one column per state.  The closed-loop poles are two complex pairs, at $-31.2 \pm 28.0j$ and $-5.46 \pm 3.58j$, all in the left-half plane.
The control torque is $\tau = 0.060\,\dot{\theta} + 0.18\,\theta - 0.078\,\dot{\beta} - 1.14\,\beta$.  When the pendulum leans, $\beta > 0$ means it leans against the direction of increasing $\theta$,
so the torque is negative: the arm is driven to get underneath it.  The gain on the arm angle has the "wrong" sign, $\tau$ increases with $\theta$.  As in the cart-pole, this is a property of the system: to move the
arm to one side you must first drive it the other way, to tilt the pendulum so that you can catch it.

## Converting open-loop model to closed-loop

We have already built the feedback path into the model, with a gain of zero.  To close the loop we just set the value of the gain block.
As a check, we can then linearize the closed-loop diagram.  Its poles should be the closed-loop poles that the design predicted

In [ ]:
Kgain.K = K  # set the gain block to the computed LQR gain
lin_cl = bd.linearize(x=[0, 0, 0, math.pi], u=[0], inputs=["torque"], outputs=[theta, alpha], full=True)
print(lin_cl.poles)
print(all(np.min(np.abs(poles_cl - p)) < 1e-6 for p in lin_cl.poles))

The unstable pole has been moved into the left-half plane.

We simulate the closed-loop system for 5 seconds, with the same torque pulse as before, from the same upright start.  We can observe:
* the pendulum no longer falls over.  The pendulum angle never exceeds 0.031 rad (1.7°) from upright and returns to it, whereas without feedback the same pulse knocked it over,
* the arm turns by at most 0.081 rad (4.6°) and returns to its starting angle.  About 0.6 s after the pulse starts the pendulum is back within 5 mrad of upright and the arm within 20 mrad of its start,
* the torque on the arm, the pulse plus the feedback, never exceeds 0.0204 N m in magnitude, well within the 0.09 N m estimate of the motor's limit.  The feedback opposes the pulse and cancels most of it within about 0.1 s.  When the pulse ends the feedback is left alone, and shows as a brief reversal to $-0.0204$ N m on the torque scope, before dying away,
* the energy stays within 1% of its upright value, $2 m_p g l_p = 0.0306$ J, since the pendulum never leaves the vertical.

In [ ]:
Kgain.K = K  # the loop is closed
out = sim.run(bd, T=5, watch=[tau, theta, alpha])  # simulate for 5s

## Exercises for the reader

* Limit the torque to $\pm 0.09$ N m with a `CLIP` block, and find how far from upright the pendulum can start and still be caught.
  * The linear controller's initial torque for a start $\beta_0$ from upright is $K_\beta \beta_0$, so you can predict the answer.  How close is it?
  * How does it change if you make $R$ larger, to ask for less torque?  What happens to the arm?
* Add a setpoint for the arm angle, a step of 0.2 rad, so that the control law becomes $\tau = -\mathbf{K}(x - x_d)$.
  * You will see the arm move briefly in the wrong direction before moving to the new angle.  How large is this effect?
* Swing the pendulum up from hanging, $\alpha(0) = 0$, with an energy-based controller.  To start it hanging, give the `alpha` integrator `x0=0`, or for a diagram you have already built set `bd["alpha"].x0 = np.array([0.0])`.
  * The energy of the upright pendulum at rest is $2 m_p g l_p$.  Equation (5) says that the torque changes the energy directly only through $\tau \dot{\theta}$, so the controller has to use the arm to pump up the pendulum's swing.  Look for a rule that times the torque to the pendulum's motion, in proportion to the energy error, and stops when $E$ reaches $2 m_p g l_p$.
  * Switch to the LQ controller when the pendulum is close enough to upright.  The wrapped angle $\beta$ is what makes this possible.
* We assumed a perfect torque actuator:
  * Replace it with a DC motor model.  The torque is $k_t (V - k_m \dot{\theta}) / R_m$ for an input voltage $V$ and back-emf constant $k_m$, which also damps the arm, and the voltage is limited to $\pm 18$ V.
  * A stepper motor, or a servo with a fast velocity loop, is closer to a velocity source: the arm angle is prescribed and only equation (4) remains, with $\ddot{\theta}$ as the input.  This is almost exactly the cart-pole's pendulum, with the arm acceleration playing the part of the cart's.  Design a controller for that.
* Investigate model error: change the pendulum length, the friction coefficients or the arm inertia in the plant but not in the controller.
  * The arm inertia is the least certain parameter, see the sources above.  Use $J_r = 5.7 \times 10^{-5}$ kg m², Table 1 taken literally, in the plant.  Is the controller still stable?
* How much does the pendulum's rotation about the vertical axis matter?  Replace $I_p \sin^2 \alpha$ by $m_p l_p^2 \sin^2 \alpha$ in the model, as Quanser does, and compare the swing-up.
* Identify the parameters of your own pendulum.  With the arm held still, the small oscillations of the hanging pendulum have the frequency $\sqrt{m_p g l_p / I_p}$, which is 10.6 rad/s for the values here, and decay at a rate set by $D_p$.  Measure both and compare.